In [2]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
df =  pd.read_parquet("all_teams_last10seasons_with_opponent_rolls.parquet")

In [3]:
target = 'TARGET_WL'
exclude_cols = [
    'Team_ID',      # Note: case-sensitive
    'Team_ID_opp',  # Also exclude opponent ID
    'Game_ID',      # Note: case-sensitive
    target
]

# Create features list by excluding the right columns
features = [col for col in df.columns if col not in exclude_cols]

In [4]:
df_model = df.dropna(subset=features + [target])

# Split chronologically (e.g., 80% train, 20% test)
split_idx = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_idx]
test_df = df_model.iloc[split_idx:]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

In [5]:
bool_cols = X_train.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X_train[bool_cols] = X_train[bool_cols].astype(int)
    X_test[bool_cols] = X_test[bool_cols].astype(int)

/var/folders/14/drwtsk3n2wvdsmw2rj7tqh040000gn/T/ipykernel_82010/2585570748.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[bool_cols] = X_train[bool_cols].astype(int)
/var/folders/14/drwtsk3n2wvdsmw2rj7tqh040000gn/T/ipykernel_82010/2585570748.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[bool_cols] = X_test[bool_cols].astype(int)


In [12]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1️⃣ Standardize Data ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2️⃣ Fit Full PCA and Inspect Variance ---
pca_full = PCA()
pca_full.fit(X_train_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

print(f"\n Current Feature Count: {X_train.shape[1]}")


print(f"\n Variance Explained by Components:")
print(f"  First 5 components: {cumulative_variance[4]:.1%}")
print(f"  First 10 components: {cumulative_variance[9]:.1%}")
print(f"  First 20 components: {cumulative_variance[19]:.1%}")

n_components_80 = np.argmax(cumulative_variance >= 0.80) + 1
n_components_90 = np.argmax(cumulative_variance >= 0.85) + 1
print(f"  Number of PCs for 80% variance: {n_components_80}")
print(f"  Number of PCs for 90% variance: {n_components_90}")

# --- 3️⃣ Fit Optimal PCA and Inspect Top Features ---
n_components_optimal = n_components_90
pca_optimal = PCA(n_components=n_components_optimal)
pca_optimal.fit(X_train_scaled)

loadings = pd.DataFrame(
    pca_optimal.components_.T,
    columns=[f'PC{i+1}' for i in range(n_components_optimal)],
    index=features
)

print(f"\n📋 Top Features for First 5 PCs:")
print("="*60)
for i in range(min(5, n_components_optimal)):
    pc_name = f'PC{i+1}'
    print(f"\n{pc_name} (explains {pca_optimal.explained_variance_ratio_[i]:.1%} of variance):")
    top_loadings = loadings[pc_name].abs().sort_values(ascending=False).head(10)
    for feature in top_loadings.index:
        print(f"  {feature:40s}: {loadings.loc[feature, pc_name]:>7.3f}")


 Current Feature Count: 53

 Variance Explained by Components:
  First 5 components: 35.4%
  First 10 components: 52.7%
  First 20 components: 74.1%
  Number of PCs for 80% variance: 24
  Number of PCs for 90% variance: 27

📋 Top Features for First 5 PCs:

PC1 (explains 8.4% of variance):
  FG_PCT_DIFF                             :   0.401
  WIN_PCT_DIFF                            :   0.352
  EFG_PCT_rolling5                        :   0.302
  EFG_PCT_rolling5_opp                    :  -0.287
  PTS_rolling5_opp                        :  -0.260
  TEAM_LAST10_WINS_opp                    :  -0.249
  PTS_rolling5                            :   0.245
  FG3_PCT_rolling5                        :   0.241
  TEAM_LAST10_WINS                        :   0.240
  FG3_PCT_rolling5_opp                    :  -0.230

PC2 (explains 8.1% of variance):
  GAMES_IN_LAST_14                        :   0.361
  GAMES_IN_LAST_14_opp                    :   0.359
  PACE_MATCHUP                            :  -0.310

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# --- 4️⃣ Transform Data into Principal Components ---
X_train = pca_optimal.transform(X_train_scaled)
X_test = pca_optimal.transform(X_test_scaled)

print(f"Shape of training data after PCA: {X_train.shape}")
print(f"Shape of test data after PCA: {X_test.shape}")


log_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

Shape of training data after PCA: (7931, 27)
Shape of test data after PCA: (1983, 27)


,steps,"[('scaler', ...), ('logreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import statsmodels.api as sm
import numpy as np

# Add constant (intercept)
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit logistic regression
model = sm.Logit(y_train, X_train_sm).fit()

# Predict probabilities (not classes yet)
y_proba = model.predict(X_test_sm)

# Convert probabilities to class predictions (threshold = 0.5)
y_pred_log = (y_proba >= 0.5).astype(int)

print("\nAIC:", model.aic.round(2))
print("BIC:", model.bic.round(2))
print("\nAccuracy:", round(accuracy_score(y_test, y_pred_log),2))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))


Optimization terminated successfully.
         Current function value: 0.651407
         Iterations 5

AIC: 10388.62
BIC: 10584.02

Accuracy: 0.63

Confusion Matrix:
 [[825 381]
 [343 434]]

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.68      0.70      1206
           1       0.53      0.56      0.55       777

    accuracy                           0.63      1983
   macro avg       0.62      0.62      0.62      1983
weighted avg       0.64      0.63      0.64      1983



In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

print("\n=== SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


=== SVM ===
Accuracy: 0.6111951588502269
Confusion Matrix:
 [[768 438]
 [333 444]]
              precision    recall  f1-score   support

           0       0.70      0.64      0.67      1206
           1       0.50      0.57      0.54       777

    accuracy                           0.61      1983
   macro avg       0.60      0.60      0.60      1983
weighted avg       0.62      0.61      0.61      1983



In [16]:
# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\n=== Random Forest ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 2))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


=== Random Forest ===
Accuracy: 0.61
Confusion Matrix:
 [[797 409]
 [355 422]]
              precision    recall  f1-score   support

           0       0.69      0.66      0.68      1206
           1       0.51      0.54      0.52       777

    accuracy                           0.61      1983
   macro avg       0.60      0.60      0.60      1983
weighted avg       0.62      0.61      0.62      1983



In [19]:
import os
# --- Ensure output folder exists ---
output_folder = "tables"
os.makedirs(output_folder, exist_ok=True)

# --- Collect all model metrics ---
results = []

# Logistic Regression
results.append({
    "model": "Logistic Regression",
    "accuracy": round(accuracy_score(y_test, y_pred_log), 2),
    "precision": round(precision_score(y_test, y_pred_log, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_log, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_log, average='macro'), 2)
})

# SVM
results.append({
    "model": "SVM",
    "accuracy": round(accuracy_score(y_test, y_pred_svm), 2),
    "precision": round(precision_score(y_test, y_pred_svm, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_svm, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_svm, average='macro'), 2)
})

# Random Forest
results.append({
    "model": "Random Forest",
    "accuracy": round(accuracy_score(y_test, y_pred_rf), 2),
    "precision": round(precision_score(y_test, y_pred_rf, average='macro'), 2),
    "recall": round(recall_score(y_test, y_pred_rf, average='macro'), 2),
    "f1_score": round(f1_score(y_test, y_pred_rf, average='macro'), 2)
})

# --- Convert to DataFrame and save ---
comparison_table = pd.DataFrame(results)
comparison_table.to_csv(os.path.join(output_folder, "model_comparison.csv"), index=False)

print("✅ Model comparison table saved to tables/model_comparison.csv")
print(comparison_table)


✅ Model comparison table saved to tables/model_comparison.csv
                 model  accuracy  precision  recall  f1_score
0  Logistic Regression      0.63       0.62    0.62      0.62
1                  SVM      0.61       0.60    0.60      0.60
2        Random Forest      0.61       0.60    0.60      0.60
